# viz

> mempool.space-style block cards: per-tx feerate histograms, summary stats, and a recent-block fee-rate sparkline.

In [ ]:
#| default_exp viz

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.figure import Figure

from slashbtc.core import Chain
from slashbtc.transform import block_fee_rows

## Per-tx feerates

For each non-coinbase tx in the block we want `(vsize, feerate_sat_per_vB)`. We use `getblock` verbosity 3 — it returns each input's prevout inline, so fee = sum(prevouts) − sum(outputs). Requires Bitcoin Core ≥ 25.


In [ ]:
#| export
_BLOCK_CACHE: dict[str, tuple[dict, list[tuple[int, float]]]] = {}

def block_feerates(
    chain: Chain, height_or_hash: int | str
) -> tuple[dict, list[tuple[int, float]]]:
    """Return `(block_dict, [(vsize, feerate_sat_per_vB), ...])` skipping coinbase.

    Memoized by input — calling twice with the same height (or hash) is free.
    Clear with `_BLOCK_CACHE.clear()` if you suspect a reorg.
    """
    key = str(height_or_hash)
    if key in _BLOCK_CACHE:
        return _BLOCK_CACHE[key]
    blk = chain.block(height_or_hash, verbosity=3)
    tx_rows = block_fee_rows(blk)
    rows = [(row["vsize_vb"], row["fee_sat_vb"]) for row in _fee_rows(tx_rows)]
    result = (blk, rows)
    _BLOCK_CACHE[key] = result
    if "hash" in blk:
        _BLOCK_CACHE[blk["hash"]] = result
    return result


## Bucketing

A block has 2k–4k txs — too many to draw one bar per tx. Bucket them into *equal-vsize chunks* sorted by feerate. Each bucket's bar length = its total vsize, color = its mean feerate. Equal-vsize chunks (rather than equal-count) make the bars proportional to block weight, matching the mempool.space style.

In [ ]:
#| export
def feerate_buckets(
    rows: Iterable[tuple[int, float]], n_bins: int = 24
) -> tuple[list[float], list[int]]:
    """Bucket txs into `n_bins` equal-vsize chunks ordered low→high feerate."""
    rows = sorted(rows, key=lambda r: r[1])  # ascending feerate
    if not rows:
        return [], []
    total = sum(v for v, _ in rows)
    target = total / n_bins
    feerates: list[float] = []
    vsizes: list[int] = []
    cur_v, cur_fw = 0, 0.0
    for v, fr in rows:
        cur_v += v
        cur_fw += fr * v
        if cur_v >= target:
            feerates.append(cur_fw / cur_v)
            vsizes.append(cur_v)
            cur_v, cur_fw = 0, 0.0
    if cur_v > 0:
        feerates.append(cur_fw / cur_v)
        vsizes.append(cur_v)
    return feerates, vsizes

## Block summary

`getblockstats` is fast (single call) and returns everything we need for the stats panel. Sparkline is just N more of the same.

In [ ]:
#| export
def block_summary(chain: Chain, height: int) -> dict:
    "Aggregate block stats from `getblockstats`."
    return chain.rpc.call("getblockstats", height)

In [ ]:
#| export
def feerate_trend(chain: Chain, end_height: int, n: int = 20) -> list[float]:
    "Median feerate (sat/vB) for the `n` blocks ending at `end_height` (inclusive)."
    heights = range(end_height - n + 1, end_height + 1)
    with ThreadPoolExecutor(max_workers=8) as pool:
        stats = list(pool.map(
            lambda h: chain.rpc.call("getblockstats", h, ["feerate_percentiles"]),
            heights,
        ))
    return [s["feerate_percentiles"][2] for s in stats]

## Rendering

Yellow→black colormap matches the screenshot: low feerate = warm, high feerate = dark. Bars are drawn high→low feerate (dark on top); flip to taste.

In [ ]:
#| export
_FEE_CMAP = LinearSegmentedColormap.from_list("fee", ["#fff5b8", "#f5a623", "#1a1a1a"])
_DIST_CMAP = LinearSegmentedColormap.from_list("fee_dist", ["#ffd89a", "#ff7a1a", "#101820"])

def _fmt_age(ts: int, now: int | None = None) -> str:
    now = now or int(datetime.now(tz=timezone.utc).timestamp())
    secs = max(0, now - ts)
    if secs < 60: return f"{secs}s ago"
    if secs < 3600: return f"{secs // 60}m ago"
    if secs < 86400: return f"{secs // 3600}h ago"
    return f"{secs // 86400}d ago"

def transaction_fee_histogram(
    rows: Iterable[dict],
    max_sat_vb: int = 200,
    bucket_width: int = 1,
    include_empty: bool = False,
) -> list[dict]:
    """Bucket DB-shaped transaction rows by floored sat/vB for charting."""
    if max_sat_vb <= 0:
        raise ValueError("max_sat_vb must be positive")
    if bucket_width <= 0:
        raise ValueError("bucket_width must be positive")

    buckets: dict[int, dict] = {}
    if include_empty:
        for lower in range(0, max_sat_vb, bucket_width):
            buckets[lower] = _empty_fee_bucket(lower, lower + bucket_width)
        buckets[max_sat_vb] = _empty_fee_bucket(max_sat_vb, None)

    for row in _fee_rows(rows):
        fee_sat_vb = float(row["fee_sat_vb"])
        lower = int(np.floor(fee_sat_vb / bucket_width) * bucket_width)
        if lower >= max_sat_vb:
            lower, upper = max_sat_vb, None
        else:
            upper = lower + bucket_width
        bucket = buckets.setdefault(lower, _empty_fee_bucket(lower, upper))
        bucket["tx_count"] += 1
        bucket["total_vsize_vb"] += row["vsize_vb"]
        bucket["total_fees_sats"] += row["fee_sats"]

    out = []
    for lower in sorted(buckets):
        bucket = buckets[lower]
        if bucket["tx_count"]:
            bucket["mean_fee_sat_vb"] = bucket["total_fees_sats"] / bucket["total_vsize_vb"]
        if include_empty or bucket["tx_count"]:
            out.append(bucket)
    return out


def transaction_fee_summary(rows: Iterable[dict], percentiles: tuple[int, ...] = (10, 25, 50, 75, 90, 99)) -> dict:
    """Return the headline fee metrics for DB-shaped transaction rows."""
    fee_rows = _fee_rows(rows)
    rates = sorted(float(row["fee_sat_vb"]) for row in fee_rows)
    positive_rates = [rate for rate in rates if rate > 0]
    tail = _overpayment_tail(rates)
    return {
        "non_coinbase_tx_count": len(rates),
        "total_vsize_vb": sum(row["vsize_vb"] for row in fee_rows),
        "total_fees_sats": sum(row["fee_sats"] for row in fee_rows),
        "min_fee_sat_vb": min(rates) if rates else None,
        "max_fee_sat_vb": max(rates) if rates else None,
        "clearing_fee_sat_vb": min(positive_rates) if positive_rates else (min(rates) if rates else None),
        "percentiles": {p: _nearest_rank(rates, p) for p in percentiles},
        "overpayment_tail": tail,
    }


def fee_distribution_chart(
    rows: Iterable[dict],
    fig: Figure | None = None,
    max_sat_vb: int = 200,
    title: str = "FEE DISTRIBUTION (sat/vB)",
) -> Figure:
    """Render a mempool.space-style fee distribution panel from DB-shaped tx rows."""
    rows = list(rows)
    summary = transaction_fee_summary(rows)
    buckets = transaction_fee_histogram(rows, max_sat_vb=max_sat_vb, include_empty=True)
    positive = [b for b in buckets if b["bucket_lower_sat_vb"] > 0 and b["tx_count"] > 0]

    fig = fig or plt.figure(figsize=(7.2, 4.8))
    gs = fig.add_gridspec(2, 1, height_ratios=[3.2, 1.35], hspace=0.12)
    ax = fig.add_subplot(gs[0, 0])
    ax_meta = fig.add_subplot(gs[1, 0])
    fig.patch.set_facecolor("white")

    if positive:
        xs = np.array([b["bucket_lower_sat_vb"] for b in positive], dtype=float)
        ys = np.array([b["tx_count"] for b in positive], dtype=float)
        norm = Normalize(vmin=1, vmax=max_sat_vb)
        colors = [_DIST_CMAP(norm(min(x, max_sat_vb))) for x in xs]
        ax.bar(xs, ys, width=np.maximum(0.8, xs * 0.045), color=colors, edgecolor="white", linewidth=0.15)
        ax.set_xscale("log")
        ax.set_xlim(0.9, max_sat_vb * 1.18)
    else:
        ax.text(0.5, 0.5, "no positive-fee transactions", ha="center", va="center", color="#6b7280", transform=ax.transAxes)
        ax.set_xlim(0, max_sat_vb)

    median = summary["percentiles"].get(50)
    if median and median > 0:
        ax.axvline(median, color="#111827", linestyle=(0, (2, 2)), linewidth=1.0)
        ymax = ax.get_ylim()[1]
        ax.text(median * 1.08, ymax * 0.93, f"median {_fmt_rate(median)} sat/vB", fontsize=10, color="#111827", va="top")

    ax.set_title(title, loc="left", fontsize=12, color="#111827", pad=14)
    ax.set_ylabel("txs", rotation=0, labelpad=24, color="#4b5563")
    ax.set_xticks([1, 5, 10, 25, 50, 100, max_sat_vb])
    ax.set_xticklabels(["1", "5", "10", "25", "50", "100", f"{max_sat_vb}+"])
    ax.tick_params(axis="both", colors="#6b7280", labelsize=9, length=0)
    ax.grid(axis="y", color="#e5e7eb", linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_visible(False)

    _draw_fee_distribution_meta(ax_meta, summary)
    return fig


def _draw_fee_distribution_meta(ax, summary: dict) -> None:
    ax.set_axis_off()
    percentiles = summary["percentiles"]
    labels = [10, 25, 50, 75, 90, 99]
    for i, p in enumerate(labels):
        x = i / len(labels) + 0.01
        ax.text(x, 0.72, f"p{p}", transform=ax.transAxes, fontsize=9, color="#6b7280")
        ax.text(x, 0.45, _fmt_rate(percentiles.get(p)), transform=ax.transAxes, fontsize=13, color="#111827")
        if i:
            ax.plot([x - 0.035, x - 0.035], [0.43, 0.78], transform=ax.transAxes, color="#e5e7eb", linewidth=1)

    ax.plot([0, 1], [0.35, 0.35], transform=ax.transAxes, color="#e5e7eb", linewidth=1)
    clearing = summary["clearing_fee_sat_vb"]
    tail = summary["overpayment_tail"]
    ax.text(0.01, 0.14, "clearing fee", transform=ax.transAxes, fontsize=9, color="#6b7280")
    ax.text(0.01, 0.00, f"{_fmt_rate(clearing)} sat/vB", transform=ax.transAxes, fontsize=13, color="#111827")
    ax.text(0.52, 0.14, "overpayment tail", transform=ax.transAxes, fontsize=9, color="#6b7280")
    ax.text(
        0.52,
        0.00,
        f"{tail['tx_count']:,} txs ({tail['share'] * 100:.1f}%)",
        transform=ax.transAxes,
        fontsize=13,
        color="#111827",
    )


def _fmt_rate(value) -> str:
    if value is None:
        return "n/a"
    value = float(value)
    if value >= 100:
        return f"{value:,.0f}"
    if value >= 10:
        return f"{value:.0f}"
    if value == 0:
        return "0"
    return f"{value:.1f}".rstrip("0").rstrip(".")


def _fee_rows(rows: Iterable[dict]) -> list[dict]:
    return [
        row for row in rows
        if not row.get("is_coinbase")
        and row.get("fee_sat_vb") is not None
        and row.get("fee_sats") is not None
        and row.get("vsize_vb")
    ]


def _empty_fee_bucket(lower: int, upper: int | None) -> dict:
    return {
        "bucket_lower_sat_vb": lower,
        "bucket_upper_sat_vb": upper,
        "tx_count": 0,
        "total_vsize_vb": 0,
        "total_fees_sats": 0,
        "mean_fee_sat_vb": None,
    }


def _nearest_rank(values: list[float], percentile: float) -> float | None:
    if not values:
        return None
    if percentile <= 0:
        return values[0]
    if percentile >= 100:
        return values[-1]
    index = int(np.ceil(percentile / 100 * len(values))) - 1
    return values[max(0, min(index, len(values) - 1))]


def _overpayment_tail(rates: list[float], share: float = 0.05) -> dict:
    positive_rates = [rate for rate in rates if rate > 0]
    if not positive_rates:
        return {"threshold_fee_sat_vb": None, "tx_count": 0, "share": 0.0}
    threshold = _nearest_rank(positive_rates, 100 * (1 - share))
    count = sum(1 for rate in positive_rates if rate >= threshold)
    return {
        "threshold_fee_sat_vb": threshold,
        "tx_count": count,
        "share": count / len(rates),
    }


In [ ]:
#| export
def block_card(
    chain: Chain,
    height: int,
    fig: Figure | None = None,
    with_sparkline: bool = True,
    btc_usd: float | None = None,
    trend: list[float] | None = None,
) -> Figure:
    """Render one mempool.space-style block row.

    Pass a precomputed `trend` (list of medians) to skip the per-card sparkline
    RPCs — useful when rendering many cards back-to-back.
    """
    _blk, rows = block_feerates(chain, height)
    s = block_summary(chain, height)
    feerates, vsizes = feerate_buckets(rows, n_bins=24)

    fig = fig or plt.figure(figsize=(13, 2.4))
    gs = fig.add_gridspec(
        1, 6,
        width_ratios=[1.4, 2.0, 1.0, 1.0, 1.4, 1.6],
        wspace=0.05,
    )

    ax_h = fig.add_subplot(gs[0, 0])
    if feerates:
        norm = Normalize(vmin=min(feerates), vmax=max(feerates))
        colors = [_FEE_CMAP(norm(fr)) for fr in feerates]
        y = np.arange(len(feerates))
        ax_h.barh(y, vsizes, color=colors, edgecolor="none", height=0.85)
        ax_h.invert_yaxis()
    ax_h.set_axis_off()

    ax_id = fig.add_subplot(gs[0, 1])
    ax_id.axis("off")
    ts = datetime.fromtimestamp(s["time"], tz=timezone.utc).strftime("%b %-d, %Y  %H:%M:%S")
    ax_id.text(0, 0.65, f"{s['height']:,}", fontsize=22, weight="bold")
    ax_id.text(0, 0.30, ts, fontsize=9, color="gray")

    ax_tx = fig.add_subplot(gs[0, 2])
    ax_tx.axis("off")
    ax_tx.text(0, 0.65, f"{s['txs']:,}", fontsize=18)
    ax_tx.text(0, 0.45, "txs", fontsize=10, color="gray")

    ax_sz = fig.add_subplot(gs[0, 3])
    ax_sz.axis("off")
    ax_sz.text(0, 0.65, f"{s['total_size']/1e6:.2f}", fontsize=18)
    ax_sz.text(0, 0.45, "MB", fontsize=10, color="gray")

    ax_fee = fig.add_subplot(gs[0, 4])
    ax_fee.axis("off")
    fee_btc = s['totalfee'] / 1e8
    ax_fee.text(0, 0.65, f"{fee_btc:.2f}", fontsize=18)
    ax_fee.text(0.45, 0.68, "BTC", fontsize=9, color="gray")
    if btc_usd is not None:
        ax_fee.text(0, 0.30, f"${fee_btc * btc_usd:,.0f}", fontsize=10, color="gray")

    ax_md = fig.add_subplot(gs[0, 5])
    ax_md.axis("off")
    median_fr = s['feerate_percentiles'][2]
    ax_md.text(0, 0.85, "median fee", fontsize=9, color="gray")
    ax_md.text(0, 0.55, f"{median_fr} sat/vB", fontsize=15, weight="bold")
    if with_sparkline:
        try:
            spark = trend if trend is not None else feerate_trend(chain, height, n=20)
            inset = ax_md.inset_axes([0, 0, 1, 0.35])
            inset.plot(spark, color="#f5a623", linewidth=1.5)
            inset.axis("off")
        except Exception:
            pass

    age = _fmt_age(s['time'])
    fig.text(0.005, 0.5, age, fontsize=10, va='center', color='gray')
    return fig

In [ ]:
#| export
def blocks_view(
    chain: Chain,
    n: int = 6,
    end_height: int | None = None,
    btc_usd: float | None = None,
    spark_window: int = 20,
    max_workers: int = 6,
) -> Figure:
    """Stack `n` block cards newest→oldest, like the mempool.space block list.

    Fetches the `n` blocks in parallel and computes the sparkline window once,
    so total RPC cost is roughly `n` big calls + `n + spark_window - 1` small.
    """
    end_height = end_height or chain.height()
    heights = [end_height - i for i in range(n)]

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        list(pool.map(lambda h: block_feerates(chain, h), heights))

    trend_low = end_height - n - spark_window + 2
    full_trend = feerate_trend(chain, end_height, n=n + spark_window - 1)

    fig = plt.figure(figsize=(13, 2.4 * n))
    subfigs = fig.subfigures(n, 1, hspace=0.0)
    if n == 1:
        subfigs = [subfigs]
    for i, sf in enumerate(subfigs):
        h = heights[i]
        offset = h - trend_low + 1
        spark = full_trend[max(0, offset - spark_window):offset]
        block_card(chain, h, fig=sf, btc_usd=btc_usd, trend=spark)
    return fig

## Demo

Tagged `notest` so `nbdev_test` skips it; un-skip when your RPC tunnel is up.

In [ ]:
#| eval: false
#| notest
chain = Chain()
fig = blocks_view(chain, n=6)
fig

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()